In [1]:
import sys
import os
from pathlib import Path
import pandas as pd

project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src import check_quality
from src import metrics_by_panel_sizes

In [2]:
path_source = os.path.join('sourcedata','preload_data.csv')
df = pd.read_csv(path_source)
df = df.drop('Unnamed: 0', axis=1 )

In [3]:
df_clean, badboys = check_quality(df)
print(badboys)
df_clean

df have no duplicated rows, checking visualisation time
set()


,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done,phase,clip_uid,MAD,scene_player
0,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,0,True,29.2107,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,3.166667,True,ppo_mario_ep-20,w3l1s1_ppo_0,NaN,w3l1s1_ppo
1,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,1,False,25.1595,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,ppo_mario_ep-20,w3l1s1_ppo_1,NaN,w3l1s1_ppo
2,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,2,False,10.2153,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,ppo_mario_ep-20,w3l1s1_ppo_2,NaN,w3l1s1_ppo
3,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,3,True,13.9311,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.400000,True,ppo_mario_ep-20,w3l1s1_ppo_3,NaN,w3l1s1_ppo
4,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,4,False,12.5782,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,ppo_mario_ep-20,w3l1s1_ppo_4,NaN,w3l1s1_ppo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118045,277659,6a260aa776e9e6b9e17d272b,w5l2s2,sub-05,global,5,False,NaN,sourcedata/sub-05/ses-004/beh/videos/sub-05_se...,NaN,NaN,Early discovery,w5l2s2_sub-05_5,NaN,w5l2s2_sub-05
118046,277659,6a260aa776e9e6b9e17d272b,w5l2s2,sub-05,global,6,True,NaN,sourcedata/sub-05/ses-004/beh/videos/sub-05_se...,NaN,NaN,Early discovery,w5l2s2_sub-05_6,NaN,w5l2s2_sub-05
118047,277659,6a260aa776e9e6b9e17d272b,w5l2s2,sub-05,global,7,True,NaN,sourcedata/sub-05/ses-004/beh/videos/sub-05_se...,NaN,NaN,Late discovery,w5l2s2_sub-05_7,NaN,w5l2s2_sub-05
118048,277659,6a260aa776e9e6b9e17d272b,w5l2s2,sub-05,global,8,True,NaN,sourcedata/sub-05/ses-004/beh/videos/sub-05_se...,NaN,NaN,Late discovery,w5l2s2_sub-05_8,NaN,w5l2s2_sub-05


In [4]:
for scene in df_clean['scene_id'].unique():
    df_f = df_clean[df_clean['scene_id']==scene]
    for player in df_f['player'].unique():
        df_p = df_f[df_f['player']==player]
        print('For scene ',scene,'and player',player, ' they are', df_p['judge_ID'].nunique(),'judges')
df_clean['scene_id'].value_counts()

For scene  w3l1s1 and player ppo  they are 50 judges
For scene  w3l1s1 and player sub-01  they are 50 judges
For scene  w7l3s4 and player sub-02  they are 53 judges
For scene  w7l3s4 and player im-sub-02  they are 53 judges
For scene  w4l1s7 and player ppo  they are 50 judges
For scene  w4l1s7 and player sub-03  they are 61 judges
For scene  w4l1s7 and player im-sub-03  they are 11 judges
For scene  w2l3s8 and player ppo  they are 50 judges
For scene  w2l3s8 and player sub-06  they are 50 judges
For scene  w5l2s2 and player im-sub-05  they are 51 judges
For scene  w5l2s2 and player sub-05  they are 51 judges


scene_id
w7l3s4    26500
w3l1s1    25000
w5l2s2    24480
w4l1s7    22570
w2l3s8    19500
Name: count, dtype: int64

In [5]:
metrics = ['dice', 'percent', 'kappa']
# Panel sizes to test
panel_sizes = [3, 5, 9, 13, 17, 21, 25]
df_clean = df_clean[df_clean['question']!= 'global']
df_clean = df_clean[df_clean['player']!= 'im-sub-03']


In [7]:
results_scene = metrics_by_panel_sizes(df_clean, metrics, ['scene_id'], [25])
path_source = os.path.join('outputdata','results_by_scene.csv')
results_scene.to_csv(path_source)

grouped = results_scene.groupby(['scene_id', 'metric'])

for scene, group in grouped:
    panel_size_max = group['panel_size'].max()
    group = group[group['panel_size']==panel_size_max]
    mean_metric = group['value'].mean()
    std_metrics = group['value'].std()
    print(scene,': ', mean_metric, '+/-', std_metrics)

Subset: ('w7l3s4',): 100%|██████████| 5/5 [00:35<00:00,  7.08s/it]

('w2l3s8', 'dice') :  0.8715133591772193 +/- 0.02917466914726745
('w2l3s8', 'kappa') :  0.8093680313639966 +/- 0.041487599346329805
('w2l3s8', 'percent') :  0.9159294871794873 +/- 0.017654556368302366
('w3l1s1', 'dice') :  0.8004917476221461 +/- 0.025609531369008242
('w3l1s1', 'kappa') :  0.7010154919111591 +/- 0.03745908489186323
('w3l1s1', 'percent') :  0.8662500000000002 +/- 0.017325974545156008
('w4l1s7', 'dice') :  0.817589868602811 +/- 0.035664245208951804
('w4l1s7', 'kappa') :  0.7343451610980688 +/- 0.04918397880105793
('w4l1s7', 'percent') :  0.8848310810810809 +/- 0.020848379806764426
('w5l2s2', 'dice') :  0.8207598520207631 +/- 0.028201531740725358
('w5l2s2', 'kappa') :  0.7474646058772528 +/- 0.039704740177665614
('w5l2s2', 'percent') :  0.8952864583333333 +/- 0.017283186070699442
('w7l3s4', 'dice') :  0.7881421905879195 +/- 0.02457021266649233
('w7l3s4', 'kappa') :  0.6905839072306729 +/- 0.035709338036625286
('w7l3s4', 'percent') :  0.8653 +/- 0.016210733443859926


In [8]:
results_player = metrics_by_panel_sizes(df_clean, metrics, ['player'], [25])
path_source = os.path.join('outputdata','results_by_player.csv')
results_player.to_csv(path_source)

grouped = results_player.groupby(['player', 'metric'])

for player, group in grouped:
    panel_size_max = group['panel_size'].max()
    group = group[group['panel_size']==panel_size_max]
    mean_metric = group['value'].mean()
    std_metrics = group['value'].std()
    print(player,': ', mean_metric, '+/-', std_metrics)

Subset: ('sub-06',): 100%|██████████| 8/8 [00:35<00:00,  4.42s/it]   

('im-sub-02', 'dice') :  0.7837068820475895 +/- 0.033684395812560414
('im-sub-02', 'kappa') :  0.6935658180120586 +/- 0.046013017006930855
('im-sub-02', 'percent') :  0.87175 +/- 0.018713348015237535
('im-sub-05', 'dice') :  0.8349047346099115 +/- 0.038043424723933454
('im-sub-05', 'kappa') :  0.7810581488276638 +/- 0.04950421064296226
('im-sub-05', 'percent') :  0.9183854166666667 +/- 0.017997414523715338
('ppo', 'dice') :  0.6576087105799775 +/- 0.056305692433248435
('ppo', 'kappa') :  0.5411775887627135 +/- 0.06589799215405051
('ppo', 'percent') :  0.8251984126984127 +/- 0.021542986077913138
('sub-01', 'dice') :  0.7514893412019599 +/- 0.03271337141950614
('sub-01', 'kappa') :  0.6263343901978273 +/- 0.04821562193428703
('sub-01', 'percent') :  0.8310500000000001 +/- 0.022811203609213817
('sub-02', 'dice') :  0.7988487817214422 +/- 0.03882535863430153
('sub-02', 'kappa') :  0.6982619111897321 +/- 0.05747191891365498
('sub-02', 'percent') :  0.8642499999999999 +/- 0.02744484827740349

In [7]:
results_question = metrics_by_panel_sizes(df_clean, metrics, ['player', 'question', 'scene_id'], panel_sizes)
path_source = os.path.join('outputdata','results_by_question.csv')
results_question.to_csv(path_source)

grouped = results_question.groupby(['question','metric'])

for question, group in grouped:
    panel_size_max = group['panel_size'].max()
    group = group[group['panel_size']==panel_size_max]
    mean_metric = group['value'].mean()
    std_metrics = group['value'].std()
    print(question,': ', mean_metric, '+/-', std_metrics)

Subset: ('sub-06', 'Q_ext', 'w2l3s8'): 100%|██████████| 40/40 [05:06<00:00,  7.65s/it]  


('Q_eff', 'dice') :  0.7809885922851313 +/- 0.17587439040371108
('Q_eff', 'kappa') :  0.6345454176585134 +/- 0.21755860457978973
('Q_eff', 'percent') :  0.8432739293139293 +/- 0.10361484655986515
('Q_exH', 'dice') :  0.6116440905499729 +/- 0.26458687536034886
('Q_exH', 'kappa') :  0.5781684545847348 +/- 0.2700944950515441
('Q_exH', 'percent') :  0.929479346846847 +/- 0.04119642979162308
('Q_exL', 'dice') :  0.6532435064935065 +/- 0.24005818170099058
('Q_exL', 'kappa') :  0.6341618955605725 +/- 0.24876262365092208
('Q_exL', 'percent') :  0.9584352286902288 +/- 0.03119483610721544
('Q_ext', 'dice') :  0.8557867176482121 +/- 0.09706925308533304
('Q_ext', 'kappa') :  0.5832593538658573 +/- 0.1660182620673473
('Q_ext', 'percent') :  0.820131406791407 +/- 0.08679904791599047
